
# Closing the Rigor Gaps in the Diffusion Pipeline

This notebook demonstrates the **rigor-gap evaluation** (`eval.py`) for a study of open-source project "bus-factor diffusion" -- what happens when a founder-only maintainer steps away.

The original evaluation script re-analyzes a 15-repo corpus across five parts (A-E). This demo reproduces the evaluation's **pure-math statistical machinery verbatim** (Wilson score confidence intervals, two-proportion z-tests, exact binomial tests) and applies it to the real numbers the full run produced, plus visualizes the full per-repo classification table (Part D).

**What this demo does NOT re-run**: the original script's Part A/B/D/E also call into `method.py` (the EXPERIMENT dependency's own pipeline, e.g. `process_repo`, `run_regressions`, `placebo_check`) against 13MB+ of raw commit history JSON. That heavy re-execution is out of scope for a lightweight demo; instead we load the **already-computed results** (`mini_demo_data.json`, a curated subset of `eval_out.json`) and re-run the evaluation's own downstream math functions on them, unmodified from `eval.py`.


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# core packages used below are pre-installed on Colab; install at Colab's exact
# versions when running locally so the environment matches
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')


In [ ]:
from __future__ import annotations

import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Z_95 = 1.959964


## Load the curated demo data

`mini_demo_data.json` is a curated subset of the full evaluation output (`eval_out.json`): the full 15-repo classification table (Part D), plus the exact inputs Parts B and E fed into their statistical tests, plus the Part A placebo-budget disclosure numbers. We fetch it from GitHub with a local-file fallback so this notebook works both on Colab and locally.


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-2/evaluation-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
data = load_data()
print(data["metadata"])
print("repo table rows:", len(data["part_D_repo_table"]))


## Config

The only tunable in this analysis is the confidence level for the Wilson intervals (`Z_LEVEL`). The original script hardcodes 95% (`Z_95 = 1.959964`); we keep that as the default here too since it is what the archived numbers below were computed at.


In [ ]:
Z_LEVEL = Z_95  # 95% two-sided z critical value, matches eval.py's Z_95 constant


## Pure-math helpers (copied verbatim from `eval.py`)

These three functions have no dependency on the artifact's raw data -- they are the same statistical primitives `eval.py` uses in Parts B and E: a Wilson score confidence interval for a binomial proportion, a two-proportion z-test, and an exact two-sided binomial test.


In [ ]:
def wilson_ci(successes: int, n: int, z: float = Z_95) -> dict:
    """Wilson score 95% CI for a binomial proportion (Wilson 1927)."""
    if n == 0:
        return {"phat": None, "low": None, "high": None, "n": 0, "successes": 0}
    phat = successes / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    halfwidth = z * math.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2)) / denom
    return {
        "phat": phat,
        "low": max(0.0, center - halfwidth),
        "high": min(1.0, center + halfwidth),
        "n": n,
        "successes": successes,
    }


def two_proportion_z_test(x1: int, n1: int, x2: int, n2: int) -> dict:
    """Two-sided pooled two-proportion z-test: this corpus's rate (1) vs.
    Avelino et al.'s published rate treated as the reference (2)."""
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = math.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    if se == 0:
        return {"p1": p1, "p2": p2, "diff_pp": (p1 - p2) * 100, "z": None, "p_value": None}
    z = (p1 - p2) / se
    p_value = math.erfc(abs(z) / math.sqrt(2))  # two-sided, standard normal
    return {"p1": p1, "p2": p2, "diff_pp": (p1 - p2) * 100, "z": z, "p_value": p_value}


def binomial_exact_two_sided_p(x: int, n: int, p0: float) -> float:
    """Exact two-sided binomial test p-value against null proportion p0."""
    from math import comb

    def pmf(k: int) -> float:
        return comb(n, k) * p0**k * (1 - p0) ** (n - k)

    p_obs = pmf(x)
    return float(sum(pmf(k) for k in range(n + 1) if pmf(k) <= p_obs + 1e-12))


## Part D: the full per-repo classification table

Each of the 15 corpus repos was classified by the original pipeline's `process_repo()` into one of: `no_tfdd` (no truck-factor-drop-to-1 event), `not_founder_only_tfdd`, `right_censored`, or a founder-only TFDD event with a post-event 18-month `survival_grade`. This table is loaded as-is from the archived evaluation output (Part D of `eval.py`, which cross-checks it against the two source JSON files row-by-row).


In [ ]:
repo_table = pd.DataFrame(data["part_D_repo_table"])
repo_table


## Part B: Wilson 95% CIs -- Avelino et al. (2019) vs. this study's own TF=1 fraction

Avelino, Constantinou, Valente & Serebrenik (ESEM 2019, arXiv:1906.08058) report that 66% of TFDDs happened in systems with truck factor 1 (TF=1), out of n=315 projects. This study's own corpus found founder-only TF=1 events among 11 detected TFDDs. We compute Wilson score 95% CIs for both and check whether they overlap -- exactly as `eval.py`'s `part_B_wilson_ci` does.


In [ ]:
avelino_inputs = data["part_B_wilson_inputs"]["avelino_et_al_2019"]
avelino_ci = wilson_ci(avelino_inputs["numerator"], avelino_inputs["n"], Z_LEVEL)

this_study_inputs = data["part_B_wilson_inputs"]["this_study"]
this_study_ci = wilson_ci(this_study_inputs["numerator"], this_study_inputs["n"], Z_LEVEL)

overlap = not (this_study_ci["high"] < avelino_ci["low"] or avelino_ci["high"] < this_study_ci["low"])

print(f"Avelino et al. 2019:  phat={avelino_ci['phat']:.3f}  95% CI=[{avelino_ci['low']:.4f}, {avelino_ci['high']:.4f}]  (n={avelino_ci['n']})")
print(f"This study:           phat={this_study_ci['phat']:.3f}  95% CI=[{this_study_ci['low']:.4f}, {this_study_ci['high']:.4f}]  (n={this_study_ci['n']})")
print(f"Intervals overlap: {overlap}")


## Part E: formal tests -- this corpus's TFDD incidence and survival vs. Avelino et al.

`eval.py`'s `part_E_survivorship_bias` formally tests this corpus's TFDD incidence rate and founder-only 18-month survival rate against Avelino et al.'s published rates, using both a two-proportion z-test and an exact binomial test.


In [ ]:
e_inputs = data["part_E_two_proportion_inputs"]
avelino_incidence = e_inputs["avelino_incidence"]
avelino_survival = e_inputs["avelino_survival"]
this_incidence = e_inputs["this_corpus_incidence"]
this_survival = e_inputs["this_corpus_survival"]

incidence_test = two_proportion_z_test(this_incidence["x"], this_incidence["n"], avelino_incidence["x"], avelino_incidence["n"])
incidence_exact_p = binomial_exact_two_sided_p(this_incidence["x"], this_incidence["n"], avelino_incidence["x"] / avelino_incidence["n"])

survival_test = two_proportion_z_test(this_survival["x"], this_survival["n"], avelino_survival["x"], avelino_survival["n"])
survival_exact_p = binomial_exact_two_sided_p(this_survival["x"], this_survival["n"], avelino_survival["x"] / avelino_survival["n"])

print("TFDD incidence:")
print(f"  this corpus {this_incidence['x']}/{this_incidence['n']}={this_incidence['x']/this_incidence['n']:.1%}  vs.  Avelino et al. {avelino_incidence['x']/avelino_incidence['n']:.1%}")
print(f"  z={incidence_test['z']:.3f}  two-prop p={incidence_test['p_value']:.2e}  exact-binomial p={incidence_exact_p:.2e}")
print()
print("Founder-only 18mo survival:")
print(f"  this corpus {this_survival['x']}/{this_survival['n']}={this_survival['x']/this_survival['n']:.1%}  vs.  Avelino et al. {avelino_survival['x']/avelino_survival['n']:.1%}")
print(f"  z={survival_test['z']:.3f}  two-prop p={survival_test['p_value']:.2e}  exact-binomial p={survival_exact_p:.2e}  (n={this_survival['n']}: essentially no power)")


## Part A: the placebo-budget resolution floor

`eval.py`'s Part A discloses a hardcoded per-repo cap in the original pipeline's placebo/window-shuffle draws: even though the module-level constant `N_PLACEBO_DRAWS=500` is what the EXPERIMENT summary cites as "500 iterations", `process_repo()` actually caps every repo's draw count at `min(N_PLACEBO_DRAWS, 20)`. The achievable permutation-test p-value resolution is `1/(k+1)` for `k` draws -- so the true floor is far coarser than the cited constant implies.


In [ ]:
a_disclosure = data["part_A_placebo_disclosure"]
claimed_floor = round(1 / (a_disclosure["N_PLACEBO_DRAWS_constant"] + 1), 6)
actual_floor = round(1 / (a_disclosure["per_repo_hard_cap"] + 1), 6)
assert claimed_floor == a_disclosure["theoretical_floor_at_claimed_500"]
assert actual_floor == a_disclosure["theoretical_floor_at_actual_cap_20"]

print(f"Cited constant N_PLACEBO_DRAWS = {a_disclosure['N_PLACEBO_DRAWS_constant']}  ->  implied p-value floor = {claimed_floor}")
print(f"Actual hardcoded per-repo cap   = {a_disclosure['per_repo_hard_cap']}  ->  TRUE p-value floor  = {actual_floor}")
print(f"The cited '500 iterations' overstates resolution by {actual_floor / claimed_floor:.1f}x.")


## Results summary

A per-repo classification bar chart (Part D) alongside the two Wilson 95% CIs (Part B) side by side.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# left: per-repo classification counts
label_map = {"no_tfdd": "no TFDD", "not_founder_only_tfdd": "TFDD, not founder-only", "right_censored": "TFDD, right-censored", None: "founder-only TFDD"}
repo_table["label"] = repo_table["error_code"].map(label_map)
counts = repo_table["label"].value_counts()
axes[0].bar(counts.index, counts.values, color="#4C72B0")
axes[0].set_ylabel("number of repos")
axes[0].set_title(f"Part D: classification of {len(repo_table)} corpus repos")
axes[0].tick_params(axis="x", rotation=30)
for tick in axes[0].get_xticklabels():
    tick.set_ha("right")

# right: Wilson 95% CIs, Avelino et al. vs. this study
labels = ["Avelino et al. 2019\n(n=315)", "This study\n(n=11)"]
phats = [avelino_ci["phat"], this_study_ci["phat"]]
lows = [avelino_ci["phat"] - avelino_ci["low"], this_study_ci["phat"] - this_study_ci["low"]]
highs = [avelino_ci["high"] - avelino_ci["phat"], this_study_ci["high"] - this_study_ci["phat"]]
axes[1].errorbar([0, 1], phats, yerr=[lows, highs], fmt="o", markersize=10, capsize=8, color="#DD8452")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(labels)
axes[1].set_xlim(-0.5, 1.5)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("TF=1 fraction of TFDDs")
axes[1].set_title(f"Part B: Wilson 95% CIs (overlap={overlap})")
axes[1].axhline(avelino_ci["phat"], color="#DD8452", linestyle=":", alpha=0.4)

plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"  Part A: true placebo p-value floor is {actual_floor} (vs. the {claimed_floor} implied by the cited constant)")
print(f"  Part B: Avelino CI [{avelino_ci['low']:.3f}, {avelino_ci['high']:.3f}] vs. this-study CI [{this_study_ci['low']:.3f}, {this_study_ci['high']:.3f}] -- overlap={overlap}")
print(f"  Part D: {len(repo_table)} repos classified; {int(repo_table['founder_only_tf1'].sum())} founder-only TF=1 events")
print(f"  Part E: TFDD incidence z={incidence_test['z']:.2f} (p={incidence_test['p_value']:.1e}); survival z={survival_test['z']:.2f} (p={survival_test['p_value']:.1e}, n={this_survival['n']})")
